<a href="https://colab.research.google.com/github/hanbiphyun/ESSA_YB/blob/main/ESAA_OB_week2_2_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#수상작리뷰 : 텍스트 분류를 위한 선형 모델의 최적화 전략
1. 분석 대상 대회 개요
대회명: Toxic Comment Classification Challenge (악성 댓글 분류)

링크 : https://www.kaggle.com/competitions/jigsaw-toxic-comment-classification-challenge

목표: 위키피디아 댓글 데이터를 분석하여 6가지 유해성 카테고리(Toxic, Severe Toxic, Obscene, Threat, Insult, Identity Hate)를 다중 분류(Multi-label Classification)함.

핵심 과제: 문맥이 파괴된 욕설, 철자 변형(예: fuck), 짧은 문장 등 비정형 텍스트 데이터의 특징을 효율적으로 추출하는 것.

2. 주요 전략 및 분석 방법 (NB-SVM 모델 중심)
보고서의 핵심인 NB-SVM(Naive Bayes - Support Vector Machine) 방식은 나이브 베이즈의 통계적 강점과 선형 모델의 분류 성능을 결합한 형태입니다.

① 특징 추출 (Feature Engineering)
우리가 수업에서 배운 TfidfVectorizer를 두 가지 차원으로 확장하여 사용했습니다.

Word N-gram: 단어 단위로 분석하여 문장의 의미적 특징을 파악 (예: "bad", "hate").

Char N-gram: 철자 단위(2~6글자)로 분석하여 단어의 변형이나 오타를 포착.

효과: "looooove"나 "h.a.t.e"처럼 단어 사이에 공백이나 기호가 있어도 철자 패턴을 통해 동일한 의미로 인식함.

② 희소 행렬 결합 (Matrix Stacking)
hstack 기능을 이용해 서로 다른 성격의 데이터 팩을 하나로 합쳤습니다.

결합 내용: [Word TF-IDF] + [Char TF-IDF]

장점: 단어가 주는 '의미'와 철자가 주는 '형태적 패턴'을 모델이 동시에 학습할 수 있게 됨.

③ 나이브 베이즈 로그 비율(Log-count Ratio) 적용
단순한 선형 회귀를 넘어, 각 단어가 특정 클래스(예: 악플)에 나타날 확률과 나타나지 않을 확률의 비율을 가중치로 미리 계산하여 입력 데이터에 곱해줍니다. 이는 모델이 어떤 단어가 '결정적'인지를 훨씬 빨리 깨닫게 만듭니다.


In [ ]:

3. 주요 구현 코드 (수정 및 복사 가능)
Python
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

# 1. 데이터 로드 및 전처리
train = pd.read_csv('train.csv').fillna('unknown')
test = pd.read_csv('test.csv').fillna('unknown')
class_names = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# 2. 단어 단위 TF-IDF (의미 분석)
word_vec = TfidfVectorizer(ngram_range=(1,1), analyzer='word', max_features=10000)
train_word = word_vec.fit_transform(train['comment_text'])

# 3. 철자 단위 TF-IDF (변형/오타 분석)
char_vec = TfidfVectorizer(ngram_range=(2,6), analyzer='char', max_features=50000)
train_char = char_vec.fit_transform(train['comment_text'])

# 4. 희소 행렬 결합 (수업 시간에 배운 hstack)
X_features = hstack([train_char, train_word])

# 5. 모델 학습 (Logistic Regression 사용)
# 데이터가 방대하므로 solver='sag'를 사용해 속도 최적화
preds = pd.DataFrame()
for label in class_names:
    model = LogisticRegression(C=4, solver='sag')
    model.fit(X_features, train[label])
    # 예측값 생성 부분 생략


4. 분석 결과 및 시사점
단순함의 승리: 복잡한 신경망(RNN/CNN) 없이도 Tfidf + Logistic Regression의 조합만으로 상위 10% 이내의 성능(AUC 0.98 이상)을 기록함.

전처리의 중요성: 텍스트 데이터에서는 모델의 복잡도보다 Tfidf의 ngram_range 설정이나 char 단위 분석 같은 특징 추출 전략이 성능에 더 큰 영향을 미침.

확장성: 이 구조는 우리가 실습한 Mercari 가격 예측뿐만 아니라 스팸 메일 차단, 뉴스 카테고리 분류 등 대부분의 기초 NLP 프로젝트에 즉시 적용 가능한 표준 모델임.
